# **INTRODUCTION TO DEEP LEARING**
___

**MATERIALS**
1. [Penerapan Multilayer Perceptron (MLP): Penjelasan dan Studi Kasus](#1)
    - [Konsep MLP](#11)
    - [Studi Kasus](#12)
2. [Penerapan Convolutional Neural Netowrk (CNN): Penjelasan dan Studi Kasus](#2)
    - [Konsep CNN](#21)
    - [Studi Kasus](#22)
3. [Penerapan MLP dan CNN pada dataset CIFAR10](#3)
    - [MLP CIFAR object classifier](#31)
    - [CNN CIFAR object classifier 1](#32)
    - [CNN CIFAR object classifier 2](#32)
4. [Kesimpulan dan Bacaan Lanjutan](#4)
    - [Bacaan Lanjutan](#41)

---

**SOURCES AND LIBRARIES**
1. Tensorflow Keras:
    - Ver. 2.12.0
    - Documentations: https://www.tensorflow.org/versions/r2.12/api_docs/python/tf
    - Github: https://github.com/tensorflow/tensorflow
2. Tensorflow Datasets:
    - Ver. 2.9.9
    - Documentations: https://www.tensorflow.org/datasets/overview
    - Github: https://github.com/tensorflow/datasets
2. Datasets
    - CIFAR10: https://www.cs.toronto.edu/~kriz/cifar.html
    - MNIST: https://www.tensorflow.org/datasets/catalog/mnist

---

#### Environment Setup
Sebelum memulai proses eksperimen pada notebook ini, dilakukan konfigurasi environment dengan mengecek ketersediaan GPU untuk komputasi serta menyiapkan path yang akan digunakan untuk penyimpanan dan pengolahan dataset.

In [ ]:
import os
import tensorflow as tf
from tensorflow.python.client import device_lib
import numpy as np
from matplotlib import pyplot as plt
import tensorflow_datasets as tfds
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.datasets import mnist, cifar10
import keras


DATA_PATH = "/app/datasets"
CKPT_PATH = "/app/checkpoints"
GLOBAL_SEED = 42
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" # Ignore TensorFlow INFO and WARNING messages

tf.keras.utils.set_random_seed(GLOBAL_SEED)
tf.config.experimental.enable_op_determinism()

# Check for GPU availability
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Available: {bool(gpus)}")

if gpus:
    tf.config.set_visible_devices(gpus, 'GPU')
    usage_devices = [gpu.name for gpu in tf.config.list_logical_devices('GPU')]

    devices = device_lib.list_local_devices()
    print("\nUsage Devices:")
    for dev in devices:
        if dev.name in usage_devices and dev.device_type == "GPU":
            print("Device:", dev.name)
            print("Description:", dev.physical_device_desc)

os.makedirs("/app/datasets", exist_ok=True)
os.makedirs("/app/checkpoints", exist_ok=True)

Sebelum memulai proses implementasi, terdapat beberapa fungsi utilitas yang digunakan untuk membantu pengolahan data, visualisasi, serta proses evaluasi model.

In [ ]:
# Utility functions

# Function to generate training and validation plots for loss and accuracy
def generate_plot(result):
    plt.figure(figsize=(12, 4))
    for idx, metric in enumerate(["Loss", "Accuracy"]):
        plt.subplot(1, 2, idx+1)
        plt.plot(result.history[metric.lower()], label=f'Training {metric}')
        plt.plot(result.history[f'val_{metric.lower()}'], label=f'Validation {metric}')
        plt.xlabel('Epoch')
        plt.ylabel(f'{metric}')
        plt.legend()
        plt.title(f'{metric} Over Time')
    plt.show()

# Function to visualize model predictions with confidence scores
def plot_predictions(model, X_vis_norm, X_vis_raw, y_true, img_shape, class_names=None, cmap="gray", title="Predictions vs True Labels"):
    scores = model.predict(X_vis_norm, verbose=0)
    predictions = tf.argmax(scores, axis=1).numpy()

    n = len(X_vis_norm)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 4))
    if n == 1:
        axes = [axes]

    for i in range(n):
        axes[i].imshow(X_vis_raw[i].reshape(img_shape), cmap=cmap)

        pred_id = predictions[i]
        true_id = int(y_true[i])

        # Convert to names if provided
        if class_names is not None:
            pred_label = class_names[pred_id]
            true_label = class_names[true_id]
        else:
            pred_label = pred_id
            true_label = true_id

        conf = scores[i][pred_id] * 100

        axes[i].set_title(
            f"Conf: {conf:.2f}%\n"
            f"Pred: {pred_label}\n"
            f"True: {true_label}"
        )
        axes[i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


#function to visualize random samples from the dataset
def visualize_data(x, y, title, n=6):
    idx = np.random.choice(len(x), n, replace=False)

    plt.figure(figsize=(10,4))
    for i, j in enumerate(idx):
        plt.subplot(2,6,i+1)
        plt.imshow(x[j], cmap="gray")
        plt.title(int(y[j]))
        plt.axis("off")
    plt.suptitle(title)
    plt.show()
    
#function to visualize random samples from the dataset with class names
def class_distribution(y_or_ds, name, num_classes=10):
    if isinstance(y_or_ds, tf.data.Dataset):
        y_list = []
        for _, label in y_or_ds:
            y_list.append(int(label.numpy()))
        y = np.array(y_list)
    else:
        y = np.array(y_or_ds).reshape(-1).astype(int)

    counts = np.bincount(y, minlength=num_classes)
    total = len(y)

    print(f"\n{name} class distribution:")
    for i in range(num_classes):
        print(f"Class {i}: {counts[i]:5d}  ({counts[i]/total:.3f})")


<a id="1"></a>
## **1. Penerapan Multilayer Perceptron (MLP): Penjelasan dan Studi Kasus**
---

<a id=11></a>
### **1.1 Konsep MLP**

*Multilayer Perceptron (MLP)* merupakan salah satu arsitektur feedforward neural network yang terdiri dari beberapa lapisan (layer) fully connected atau dense layer.

Sesuai dengan namanya, MLP terdiri dari banyak lapisan neuron (perceptron). Secara umum, MLP memiliki tiga jenis layer, yaitu:

**Struktur MLP**

1. **Input Layer:** 
Input Layer terdiri dari neuron yang merepresentasikan setiap fitur dalam data. Misalnya, jika teradapat data gambar 28x28 pixel, maka input layer akan memiliki 784 neuron.

2. **Hidden Layer:**
Hidden layer terdiri dari sejumlah neuron yang melakukan proses komputasi terhadap data menggunakan operasi linear dan fungsi aktivasi non-linear. Sebuah MLP dapat memiliki satu atau lebih hidden layer. Jika jumlah hidden layer dari MLP melebihi satu, maka dapat digolongkan dalam kategori Deep Neural Network.

3. **Output Layer:** 
Output Layer merupakan layer yang mengahsilkan prediksi akhir dari jaringan. Jumlah neuron yang ada pada output layer bergantung apda jenis permaslahan yang dihadapi.

Setiap neuron dalam MLP dihubungkan dengan semua neuron di layer sebelum dan
sesudahnya. Bobot dan bias dikaitkan dengan setiap koneksi ini, dan ini yang diadjust
selama proses training.


__Fungsi Aktivasi dalam MLP__

MLP biasanya menggunakan fungsi aktivasi non-linear pada setiap neuron. Ini memungkinkan
MLP untuk memodelkan hubungan kompleks dan non-linear antara fitur. Fungsi aktivasi
yang paling umum digunakan adalah ReLU, tetapi yang lain seperti sigmoid atau tanh juga
dapat digunakan.

__Proses Training MLP__

MLP dilatih menggunakan metode yang disebut backpropagation dan algoritma optimasi
seperti Stochastic Gradient Descent (SGD) atau Adam. Proses training mencakup forward
pass di mana menghasilkan prediksi, lalu menghitung loss berdasarkan input saat ini, backward pass di mana menghitung gradient loss pakai backpropagation, dan optimasi di mana bobot dan bias diperbarui berdasarkan gradient yang dihasilkan.

<a id=12></a>
### **1.2 Studi Kasus - MNIST MLP**
MNIST (Modified National Institute of Standard and Technology) adalah kumpulan data besar yang terdiri dari gambar digit tulisan tangan. MNIST sendiri berisi 60.000 gambar untuk data training dan 10.000 gambar untuk data testing. Dalam dataset ini, digit telah dinormalisasi ukurannya dan dipusatkan dalam gambar berukuran 28x28 piksel.

In [ ]:
(X_train_MNIST, y_train_MNIST), (X_test_MNIST, y_test_MNIST) = mnist.load_data()

visualize_data(X_train_MNIST, y_train_MNIST, "MNIST TRAIN")
visualize_data(X_test_MNIST,  y_test_MNIST,  "MNIST TEST")

#### Data Splitting (Train, Validation, Test)

Pada tahap ini, dataset dibagi menjadi tiga bagian utama, yaitu training set, validation set, dan test set.

1. Training set
digunakan untuk melatih model dan update weight.

2. Validation set
digunakan untuk mengevaluasi performa model selama proses pelatihan serta membantu dalam pemilihan hyperparameter.

3. Test set
digunakan untuk mengukur performa akhir model terhadap data yang benar-benar belum pernah dilihat sebelumnya.

Proses pembagian dilakukan dengan cara shuffling data training. Setelah data diacak, sebagian data dipisahkan sebagai validation set, sementara sisanya digunakan sebagai training set.

In [ ]:
VAL_SIZE_MNIST=10000
BATCH_SIZE_MNIST_MLP = 64
RANDOM_SEED_MNIST=67

np.random.seed(RANDOM_SEED_MNIST)          # make it reproducible
perm = np.random.permutation(len(X_train_MNIST))  # shuffled indices

X_train_MNIST = X_train_MNIST[perm]
y_train_MNIST = y_train_MNIST[perm]


X_val_MNIST = X_train_MNIST[-VAL_SIZE_MNIST:]
y_val_MNIST = y_train_MNIST[-VAL_SIZE_MNIST:]

X_train_MNIST = X_train_MNIST[:-VAL_SIZE_MNIST]
y_train_MNIST = y_train_MNIST[:-VAL_SIZE_MNIST]

print("Train:", X_train_MNIST.shape, y_train_MNIST.shape)
print("Val  :", X_val_MNIST.shape, y_val_MNIST.shape)
print("Test :", X_test_MNIST.shape, y_test_MNIST.shape)

<!-- #### Class Distribution -->
Sebelum melakukan proses definisi dan training, dilakukan analisis distribusi kelas pada dataset training dan testing untuk memastikan bahwa setiap kelas memiliki jumlah sampel yang seimbang.

In [ ]:
class_distribution(y_train_MNIST, "TRAIN")
class_distribution(y_test_MNIST,  "TEST")
class_distribution(y_val_MNIST,  "VAL")

Nilai piksel dalam gambar biasanya berkisar antara 0 hingga 255. Nilai-nilai ini dinormalisasi ke rentang 0 hingga 1 dengan membagi setiap piksel dengan 255. Proses ini membantu network belajar dengan lebih efektif.

In [ ]:
X_train_MNIST = X_train_MNIST.astype(np.float32) / 255.0
X_val_MNIST   = X_val_MNIST.astype(np.float32) / 255.0
X_test_MNIST  = X_test_MNIST.astype(np.float32) / 255.0

y_train_MNIST = y_train_MNIST.astype(np.int32)
y_val_MNIST   = y_val_MNIST.astype(np.int32)
y_test_MNIST  = y_test_MNIST.astype(np.int32)

#### Arsitektur Model MLP

1. **Input Layer (28x28)**  
   Layer untuk menerima citra berwarna 32×32 piksel.

2. **Flatten Layer**  
   Mengubah citra 2D menjadi vektor 1D berukuran 784.  
   setara dengan operasi NumPy:
      ```python
      x = x.reshape(-1, 28*28)
      ```
   
3. **Hidden Layer 1**  
   Melakukan ekstraksi fitur awal dari data input.
   - Dense 128, ReLU

4. **Hidden Layer 2**  
   Melakukan transformasi lanjutan untuk menyederhanakan fitur.
   - Dense 32, ReLU

5. **Output Layer (Dense 10, Softmax)**  
   Menghasilkan distribusi probabilitas kelas.
   - Dense 10, Softmax  

In [ ]:
mlp_digit_classifier_model = keras.Sequential([
    keras.layers.Input(shape=(28,28)),
    keras.layers.Flatten(), #the same as reshapeing to (784,)
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
], name="mlp_digit_classifier")

mlp_digit_classifier_model_initial_weights = mlp_digit_classifier_model.get_weights()
mlp_digit_classifier_model.summary()

#### Model Callback
Callback merupakan objek atau fungsi yang digunakan untuk memantau dan mengontrol proses training model. Dalam proses training, digunakan beberapa callback untuk meningkatkan performa dan efisiensi pelatihan model, yaitu:

1. **Early Stopping**  
   Teknik yang menghentikan proses training secara otomatis ketika performa model pada validation set tidak lagi mengalami peningkatan.  
   Callback ini membantu mencegah overfitting dan menghindari pelatihan yang tidak perlu.

2. **Model Checkpoint**  
   Callback yang menyimpan model atau bobot terbaik selama proses training berdasarkan metrik yang dipantau (misalnya validation loss atau validation accuracy).  
   Dengan callback ini, model terbaik dapat digunakan kembali tanpa perlu melakukan training ulang.


In [ ]:
mlp_digit_classifier_model.set_weights(mlp_digit_classifier_model_initial_weights)

mlp_digit_classifier_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [ ]:
mlp_digit_classifier_model_early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
mlp_digit_classifier_model_mcp = ModelCheckpoint(f'{CKPT_PATH}/mlp_digit_classifier.keras', save_best_only=True, verbose=1, monitor='val_loss')

result = mlp_digit_classifier_model.fit(
    X_train_MNIST,               
    y_train_MNIST,               
    epochs=30, 
    batch_size=BATCH_SIZE_MNIST_MLP,
    validation_data=(X_val_MNIST, y_val_MNIST), 
    callbacks=[mlp_digit_classifier_model_early_stopping, mlp_digit_classifier_model_mcp],
)

Model terbaik yang telah disimpan akan diload kembali untuk melakukan evaluasi dan prediksi pada data uji (test set).

Output dari model berupa vektor dengan panjang 10 (sesuai jumlah kelas). Setiap elemen dalam vektor tersebut merepresentasikan **probabilitas prediksi** untuk masing-masing kelas (0–9) yang dihasilkan oleh fungsi aktivasi softmax.

Sebagai contoh, jika nilai pada indeks ke-1 adalah 0.93, maka model memprediksi bahwa gambar tersebut memiliki probabilitas 93% sebagai digit "1".

Selanjutnya dilakukan evaluasi performa model terhadap data uji untuk memperoleh nilai loss dan accuracy.



In [ ]:
mlp_digit_classifier_model.load_weights(f'{CKPT_PATH}/mlp_digit_classifier.keras')

test_loss, test_acc = mlp_digit_classifier_model.evaluate(X_test_MNIST, y_test_MNIST, verbose=0)

print(f"Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
generate_plot(result)

Secara umum, MLP menghasilkan akurasi yang sangat tinggi pada data training dan cukup baik pada data validasi. Namun, terdapat selisih performa antara training dan validation yang menunjukkan generalisasi masih dapat ditingkatkan.

In [ ]:
# Load best weights
mlp_digit_classifier_model_inference=keras.models.load_model(f"{CKPT_PATH}/mlp_digit_classifier.keras")

# Take 5 samples
X_vis = X_test_MNIST[:5]
y_true = y_test_MNIST[:5]

plot_predictions(
    mlp_digit_classifier_model_inference,
    X_vis,
    X_vis,
    y_true,
    img_shape=(28,28),
    title="MNIST Predictions"
)

<a id="2"></a>
## **2. Penerapan Convolutional Neural Netowrk (CNN): Penjelasan dan Studi Kasus**
---

<a id=21></a>
### **2.1 Konsep CNN**

*Convolutional Neural Network (CNN)* adalah jenis khusus dari neural networks yang
dirancang khusus untuk memproses data dengan struktur grid seperti gambar. CNN telah
sukses besar dalam berbagai aplikasi pengenalan gambar dan video.

**Struktur CNN**
1. **Input Layer:** 
Layer ini terdiri dari neuron yang merepresentasikan setiap input fitur dalam data. Umumnya berupa gambar berbentuk B (Total / Batch image), C (Channel), H (Height), W (Width).

2. **Convolutional Layer:** 
Layer konvolusi adalah elemen inti dari CNN. Dalam layer ini, beberapa filter digunakan untuk melakukan operasi konvolusi pada input. Hasil dari operasi ini disebut feature map atau activation map.

3. **Pooling Layer:** 
Pooling layer bertujuan untuk mengurangi dimensionalitas data. Ada berbagai teknik pooling, termasuk max pooling, average pooling, dan sum pooling.

4. **Fully Connected Layer:**
Fully connected layer menghubungkan setiap neuron di layer sebelumnya ke setiap neuron di layer berikutnya, mirip dengan apa yang kita lihat di MLP. Biasanya, fully connected layer ditempatkan di akhir arsitektur CNN, dan bertugas untuk menghasilkan prediksi akhir model.



**Fungsi Aktivasi dalam CNN**

ReLU (Rectified Linear Unit) adalah fungsi aktivasi yang paling sering digunakan dalam
CNN. Fungsi ini mampu mempercepat konvergensi jaringan neural dibandingkan dengan
sigmoid atau tanh.

**Proses Training CNN**

Proses pelatihan CNN serupa dengan MLP. Kami menggunakan metode seperti
backpropagation dan algoritma optimasi seperti Adam atau SGD untuk melatih model kami.
Kita menghitung loss dengan fungsi loss seperti cross-entropy, dan mengupdate
bobot dan bias berdasarkan gradien loss dari proses backward pass.

<a id=22></a>
### **2.2 Studi Kasus - MNIST CNN**
Berbeda dengan MLP yang hanya menggunakan layer fully connected, CNN dirancang untuk memanfaatkan struktur spasial pada data citra sehingga mampu mengekstraksi fitur secara lebih efektif.

Model CNN akan dibangun menggunakan library Keras dari TensorFlow untuk melakukan klasifikasi objek dan digit. Diharapkan arsitektur ini dapat menghasilkan performa prediksi yang lebih baik dibandingkan MLP.

Berikut merupakan kode implementasi model CNN yang digunakan dalam studi kasus ini:

#### Arsitektur Model CNN (Versi 1)

1. **Input Layer (28x28)**  
   Layer untuk menerima citra berwarna 32×32 piksel.

2. **Block 1**  
   Ekstraksi fitur lokal awal dari gambar.
   - Conv2D 32, kernel 3×3, ReLU  
   - Conv2D 32, kernel 3×3, ReLU  
   - MaxPooling 2×2  

3. **Block 2**  
   Ekstraksi fitur yang lebih kompleks dan representatif.
   - Conv2D 64, kernel 3×3, ReLU  
   - Conv2D 64, kernel 3×3, ReLU  
   - MaxPooling 2×2  

4. **Block 3**  
   Ekstraksi fitur tingkat tinggi dari gambar.
   - Conv2D 128, kernel 3×3, ReLU  
   - Conv2D 128, kernel 3×3, ReLU  
   - MaxPooling 2×2  

5. **Classifier**  
   Menghasilkan prediksi kelas.
   - Flatten  
   - Dense 128, ReLU  
   - Dense 10, Softmax  

In [ ]:
# Create model
BATCH_SIZE_MNIST_CNN=64

cnn_digit_classifier_model = keras.Sequential([
    keras.Input(shape=(28,28,1)),
    # Block 1
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Block 2
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Block 3
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Classifier
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
], name="cnn_digit_classifier")

cnn_digit_classifier_model_initial_weights = cnn_digit_classifier_model.get_weights()
cnn_digit_classifier_model.summary()

In [ ]:
cnn_digit_classifier_model.set_weights(cnn_digit_classifier_model_initial_weights)

cnn_digit_classifier_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
    ,jit_compile=False
)

In [ ]:
cnn_digit_classifier_model_early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
cnn_digit_classifier_model_mcp = ModelCheckpoint(f'{CKPT_PATH}/cnn_digit_classifier.keras', save_best_only=True, verbose=1, monitor='val_loss')

# Train model with history tracking
result = cnn_digit_classifier_model.fit(
    X_train_MNIST,               
    y_train_MNIST,               
    epochs=30, 
    batch_size=BATCH_SIZE_MNIST_CNN,
    validation_data=(X_val_MNIST, y_val_MNIST), 
    callbacks=[cnn_digit_classifier_model_early_stopping, cnn_digit_classifier_model_mcp],
)

In [ ]:
cnn_digit_classifier_model.load_weights(f'{CKPT_PATH}/cnn_digit_classifier.keras')
test_loss, test_acc = cnn_digit_classifier_model.evaluate(X_test_MNIST, y_test_MNIST, verbose=0)

print(f"Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
generate_plot(result)

Secara umum, CNN menghasilkan akurasi yang lebih tinggi dan lebih konsisten pada data validasi. Selisih antara training dan validation juga lebih kecil, model menunjukkan kemampuan generalisasi yang lebih baik dibandingkan MLP.

In [ ]:
# Take 5 samples
cnn_digit_classifier_inference_model=keras.models.load_model(f"{CKPT_PATH}/cnn_digit_classifier.keras")

X_vis = X_test_MNIST[:5]
y_true = y_test_MNIST[:5]

plot_predictions(
    cnn_digit_classifier_inference_model,
    X_vis,
    X_vis,
    y_true,
    img_shape=(28,28),
    title="MNIST Predictions"
)

<a id="3"></a>
## **3. Penerapan MLP dan CNN pada data CIFAR-10**
---

<a id=31></a>
### **3.1 MLP CIFAR object classifier**

CIFAR-10 (Canadian Institute for Advanced Research) adalah dataset yang berisi kumpulan gambar berwarna untuk keperluan klasifikasi. Dataset ini terdiri dari **60.000 gambar berwarna 32×32 piksel**, yang terbagi menjadi **50.000 gambar untuk training** dan **10.000 gambar untuk testing**.  

Gambar-gambar ini dibagi menjadi **10 kelas** yang merepresentasikan objek umum, yaitu: pesawat terbang, mobil, burung, kucing, rusa, anjing, katak, kuda, kapal, dan truk.

Untuk memudahkan loading dan processing dataset digunakan **TensorFlow Datasets (TFDS)**. TFDS adalah library tensorflow yang menyediakan kumpulan dataset siap pakai untuk keperluan machine learning dan deep learning. Selain itu, TFDS memudahkan proses pengunduhan, pemrosesan, dan pemuatan dataset dalam format yang kompatibel dengan TensorFlow.

In [ ]:

(train_ds_CIFAR, test_ds_CIFAR), info = tfds.load(
    "cifar10",
    split=["train", "test"],
    as_supervised=True,
    with_info=True,
    data_dir="/app/datasets/tfds",
)

CIFAR_CLASS = info.features["label"].names
X_vis_train_CIFAR = []
y_vis_train_CIFAR = []
X_vis_test_CIFAR = []
y_vis_test_CIFAR = []
for img, label in train_ds_CIFAR.take(10):
    X_vis_train_CIFAR.append(img)
    y_vis_train_CIFAR.append(label)
for img, label in test_ds_CIFAR.take(10):
    X_vis_test_CIFAR.append(img)
    y_vis_test_CIFAR.append(label)
visualize_data(X_vis_train_CIFAR, y_vis_train_CIFAR, "CIFAR TRAIN")
visualize_data(X_vis_test_CIFAR, y_vis_test_CIFAR, "CIFAR TEST")

train_ds_CIFAR dan test_ds_CIFAR adalah objek tf.data.Dataset, yaitu pipeline data TensorFlow yang berisi pasangan (image, label) dan dirancang untuk digunakan langsung dalam proses training model.

In [ ]:
VAL_SIZE_CIFAR=10000
BATCH_SIZE_CIFAR=64
RANDOM_SEED_CIFAR=67


train_ds_CIFAR = train_ds_CIFAR.shuffle(10000, seed=RANDOM_SEED_CIFAR, reshuffle_each_iteration=False)

# Split
val_ds_CIFAR   = train_ds_CIFAR.take(VAL_SIZE_CIFAR)
train_ds_CIFAR = train_ds_CIFAR.skip(VAL_SIZE_CIFAR)

Selanjutnya, kembali dilakukan analisis distribusi kelas pada dataset training dan testing untuk memastikan bahwa setiap kelas memiliki jumlah sampel yang seimbang.

In [ ]:
class_distribution(train_ds_CIFAR, "TRAIN")
class_distribution(test_ds_CIFAR,  "TEST")
class_distribution(val_ds_CIFAR,  "VAL")

Dengan menggunakan TFDS (TensorFlow Datasets), beberapa langkah preprocessing dan optimisasi dapat dilakukan dengan mudah dan efisien. Berikut merupakan beberapa proses yang akan dilakukan apda dataset.

**1. Preprocessing (preprocess_ds_CIFAR):**
- Melakukan normalisasi input (x/255.0) dan casting label ke int32
- Menyiapkan data agar kompatibel dengan model (float32 input, int32 label)

**2 .Mapping:**
- Menerapkan fungsi preprocess_ds_CIFAR ke setiap elemen dataset
- otomatis memanfaatkan CPU untuk mempercepat preprocessing

**3. Batching:**
- Mengelompokkan dataset menjadi batch sesuai batch size
- Setiap batch akan diproses model dalam satu forward-backward pass
- Memastikan input model tetap konsisten ukuran batch-nya

**4. Prefetching:**
- Memuat batch berikutnya di background saat model sedang training batch sekarang
- Mengurangi GPU/CPU idle time dan meningkatkan throughput pipeline

In [ ]:
def preprocess_ds_CIFAR(x, y):
    x_new = tf.cast(x, tf.float32) / 255.0
    y_new = tf.cast(y, tf.int32)
    # print(f"Change shape from {x.shape} to {x_new.shape}")
    return x_new, y_new

# train_ds_CIFAR = tf.data.Dataset.from_tensor_slices((X_train_CIFAR, y_train_CIFAR))
train_ds_CIFAR = train_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_CIFAR = train_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
train_ds_CIFAR = train_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

# val_ds_CIFAR = tf.data.Dataset.from_tensor_slices((X_val_CIFAR, y_val_CIFAR))
val_ds_CIFAR = val_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_CIFAR = val_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
val_ds_CIFAR = val_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

# test_ds_CIFAR = tf.data.Dataset.from_tensor_slices((X_test_CIFAR, y_test_CIFAR))
test_ds_CIFAR = test_ds_CIFAR.map(preprocess_ds_CIFAR, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_CIFAR = test_ds_CIFAR.batch(BATCH_SIZE_CIFAR)
test_ds_CIFAR = test_ds_CIFAR.prefetch(tf.data.AUTOTUNE)

#### Arsitektur Model MLP

1. **Input Layer (32×32×3)**  
   Layer untuk menerima citra berwarna 32×32 piksel.

2. **Flatten Layer**  
   Mengubah citra 2D menjadi vektor 1D berukuran 3072.  
   
3. **Hidden Layer 1**  
   Melakukan ekstraksi fitur awal dari data input.
   - Dense 128, ReLU

4. **Hidden Layer 2**  
   Melakukan transformasi lanjutan untuk menyederhanakan fitur.
   - Dense 32, ReLU

5. **Output Layer (Dense 10, Softmax)**  
   Menghasilkan distribusi probabilitas kelas.
   - Dense 10, Softmax  

In [ ]:
# Create model
mlp_object_classifier_model = keras.Sequential([
    keras.layers.Input(shape=(32,32,3)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
], name="mlp_object_classifier")
mlp_object_classifier_model_initial_weights = mlp_object_classifier_model.get_weights()
mlp_object_classifier_model.summary()


In [ ]:

mlp_object_classifier_model.set_weights(mlp_object_classifier_model_initial_weights)

mlp_object_classifier_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
    jit_compile=False
)
mlp_object_classifier_model_early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
mlp_object_classifier_model_mcp = ModelCheckpoint(f'{CKPT_PATH}/mlp_object_classifier.keras', save_best_only=True, verbose=1, monitor='val_loss')

result = mlp_object_classifier_model.fit(
    train_ds_CIFAR,
    epochs=30, 
    validation_data=val_ds_CIFAR, 
    callbacks=[mlp_object_classifier_model_early_stopping, mlp_object_classifier_model_mcp],
)

In [ ]:
# Plot training history
generate_plot(result)

Untuk CIFAR, MLP menghasilkan akurasi yang kurang tinggi dalam training dan validasi karena model ini tidak bisa menangkap pola spasial dalam gambar yang lebih kompleks secara efektif. 

In [ ]:
# Evaluate model
mlp_object_classifier_model.load_weights(f'{CKPT_PATH}/mlp_object_classifier.keras')
test_loss_CIFAR, test_acc_CIFAR = mlp_object_classifier_model.evaluate(test_ds_CIFAR)
print(f"Test Accuracy: {test_acc_CIFAR * 100:.2f}%")


In [ ]:
mlp_object_classifier_inference_model=keras.models.load_model(f"{CKPT_PATH}/mlp_object_classifier.keras")

num_samples = 5
X_vis, y_true = [], []

for img, lbl in test_ds_CIFAR.unbatch().take(num_samples):
    X_vis.append(img)
    y_true.append([lbl])

X_vis = tf.stack(X_vis)
X_vis_norm = tf.cast(X_vis, tf.float32)
y_true = tf.stack(y_true)
print(X_vis.shape)
print(y_true.shape)

# Now plot
plot_predictions(
    mlp_object_classifier_inference_model,
    X_vis_norm,
    X_vis.numpy(),
    y_true,
    class_names=CIFAR_CLASS,
    img_shape=(32,32,3),
    title="CIFAR MLP Predictions"
)

<a id=32></a>
### **3.2 CNN CIFAR object classifier 1**

#### Arsitektur Model CNN (Versi 1)

1. **Input Layer (32×32×3)**  
   Layer untuk menerima citra berwarna 32×32 piksel.

2. **Block 1**  
   Ekstraksi fitur lokal awal dari gambar.
   - Conv2D 32, kernel 3×3, ReLU  
   - Conv2D 32, kernel 3×3, ReLU  
   - MaxPooling 2×2  

3. **Block 2**  
   Ekstraksi fitur yang lebih kompleks dan representatif.
   - Conv2D 64, kernel 3×3, ReLU  
   - Conv2D 64, kernel 3×3, ReLU  
   - MaxPooling 2×2  

4. **Block 3**  
   Ekstraksi fitur tingkat tinggi dari gambar.
   - Conv2D 128, kernel 3×3, ReLU  
   - Conv2D 128, kernel 3×3, ReLU  
   - MaxPooling 2×2  

5. **Classifier**  
   Menghasilkan prediksi kelas.
   - Flatten  
   - Dense 128, ReLU  
   - Dense 10, Softmax  

In [ ]:
cnn_object_classifier_model = keras.Sequential([
    keras.Input(shape=(32,32,3)),

    # Block 1
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Block 2
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Block 3
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.MaxPool2D(2),
    # Classifier
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
], name="cnn_object_classifier")

cnn_object_classifier_model_initial_weights = cnn_object_classifier_model.get_weights()
cnn_object_classifier_model.summary()

In [ ]:
# Reset weights
cnn_object_classifier_model.set_weights(cnn_object_classifier_model_initial_weights)

# Compile model
cnn_object_classifier_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
    jit_compile=False
)
# Define callbacks
cnn_object_classifier_model_early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
cnn_object_classifier_model_mcp = ModelCheckpoint(f'{CKPT_PATH}/cnn_object_classifier.keras', save_best_only=True, verbose=1, monitor='val_loss')

# Train model with history tracking
result = cnn_object_classifier_model.fit(
    train_ds_CIFAR,
    epochs=30, 
    validation_data=val_ds_CIFAR, 
    callbacks=[cnn_object_classifier_model_early_stopping, cnn_object_classifier_model_mcp],
)

In [ ]:
# Plot training history
generate_plot(result)

Untuk CIFAR, CNN mampu menangkap pola spasial dalam gambar, tetapi model yang terlalu kompleks mudah mengalami overfitting, sehingga akurasi pada data training sangat tinggi sementara performa pada data test menurun drastis. 

Overfitting ini menunjukkan bahwa model terlalu menghafalkan fitur data training dan gagal menggeneralisasi ke data baru. 
Strategi seperti mengurangi jumlah layer, menambahkan dropout, dan menerapkan regularisasi dapat membantu mengurangi overfitting dan meningkatkan generalisasi model.

In [ ]:
# Evaluate model
cnn_object_classifier_model.load_weights(f'{CKPT_PATH}/cnn_object_classifier.keras')
test_loss_CIFAR, test_acc_CIFAR = cnn_object_classifier_model.evaluate(test_ds_CIFAR)
print(f"Test Accuracy: {test_acc_CIFAR * 100:.2f}%")

In [ ]:
cnn_object_classifier_inference_model=keras.models.load_model(f"{CKPT_PATH}/cnn_object_classifier.keras")

num_samples = 5
X_vis, y_true = [], []

for img, lbl in test_ds_CIFAR.unbatch().take(num_samples):
    X_vis.append(img)
    y_true.append([lbl])

X_vis = tf.stack(X_vis)
X_vis_norm = tf.cast(X_vis, tf.float32)
y_true = tf.stack(y_true)
print(X_vis.shape)
print(y_true.shape)

# Now plot
plot_predictions(
    cnn_object_classifier_inference_model,
    X_vis_norm,
    X_vis.numpy(),
    y_true,
    img_shape=(32,32,3),
    class_names=CIFAR_CLASS,
    cmap=None,
    title="CIFAR CNN Predictions"
)


<a id=33></a>
### **3.3 CNN CIFAR object classifier 2**

#### Arsitektur Model CNN (Versi 2 – Dengan BatchNorm & Dropout)

1. **Input Layer (32×32×3)**  
   Layer untuk menerima citra berwarna 32×32 piksel.

2. **Block 1**  
   - Conv2D 32, kernel 3×3, ReLU
   - BatchNormalization  
   - Conv2D 32, kernel 3×3, ReLU 
   - BatchNormalization  
   - MaxPooling 2×2  
   - Dropout 0.2  
   Ekstraksi fitur lokal dan menstabilkan distribusi aktivasi.

3. **Block 2**  
   - Conv2D 64, kernel 3×3, ReLU
   - BatchNormalization  
   - Conv2D 64, kernel 3×3, ReLU
   - BatchNormalization  
   - MaxPooling 2×2  
   - Dropout 0.4  
   Ekstraksi fitur kompleks dengan regularisasi tambahan.

4. **Block 3**  
   - Conv2D 128, kernel 3×3, ReLU
   - BatchNormalization  
   - Conv2D 128, kernel 3×3, ReLU
   - BatchNormalization  
   - MaxPooling 2×2  
   - Dropout 0.5  
   Ekstraksi fitur abstrak dengan regularisasi untuk mengurangi overfitting.

5. **Classifier**  
   - Flatten  
   - Dense 128, ReLU → BatchNormalization
   - Dropout 0.5  
   - Dense 10, Softmax  
   Menghasilkan prediksi probabilitas kelas CIFAR.

**Dropout** dan **Batch Normalization** merupakan teknik yang sering digunakan dalam CNN untuk membantu proses training menjadi lebih stabil dan mencegah overfitting:

1. **Dropout** adalah teknik regularisasi yang digunakan untuk mencegah overfitting pada neural network.  
    - Saat training, **sebagian neuron secara acak dinonaktifkan** pada setiap batch, saat inference 
    ,semua neuron digunakan, namun output biasanya dikalibrasi sesuai probabilitas dropout.
    - Tujuannya agar jaringan tidak terlalu bergantung pada neuron tertentu dan belajar **representasi fitur yang lebih general**.  

2. **Batch Normalization** menormalkan aktivasi di setiap layer untuk **mempercepat dan menstabilkan training**.  
    - Setiap batch, output layer diubah sehingga memiliki **mean ≈ 0** dan **variance ≈ 1**.  
    - Membantu model lebih tahan terhadap variasi bobot awal dan learning rate yang lebih besar.  

In [ ]:

cnn_object_classifier_model_v2 = keras.Sequential([
    keras.Input(shape=(32,32,3)),

    # Block 1
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(2),
    keras.layers.Dropout(0.2),
    # Block 2
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(2),
    keras.layers.Dropout(0.4),
    # Block 3
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(2),
    keras.layers.Dropout(0.5),
    # Classifier
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation="softmax")
], name="cnn_object_classifier_v2")

cnn_object_classifier_model_v2_initial_weights = cnn_object_classifier_model_v2.get_weights()
cnn_object_classifier_model_v2.summary()

In [ ]:
cnn_object_classifier_model_v2.set_weights(cnn_object_classifier_model_v2_initial_weights)

cnn_object_classifier_model_v2.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
    jit_compile=False
)
cnn_object_classifier_model_v2_early_stopping = EarlyStopping(monitor='val_loss', patience=8, verbose=1, mode='min')
cnn_object_classifier_model_v2_mcp = ModelCheckpoint(f'{CKPT_PATH}/cnn_object_classifier_v2.keras', save_best_only=True, verbose=1, monitor='val_loss')

result = cnn_object_classifier_model_v2.fit(
    train_ds_CIFAR,
    epochs=30, 
    validation_data=val_ds_CIFAR, 
    callbacks=[cnn_object_classifier_model_v2_early_stopping, cnn_object_classifier_model_v2_mcp],
)

In [ ]:
generate_plot(result)

Untuk CIFAR CNN versi 2 dapat terlihat menggunakan **Dropout** dan **Batch Normalization** meningkatkan kemampuan generalisasi model. Kombinasi kedua teknik ini menghasilkan model yang tidak hanya memiliki akurasi training yang tinggi, tetapi juga performa test yang lebih baik.

In [ ]:
cnn_object_classifier_model_v2.load_weights(f'{CKPT_PATH}/cnn_object_classifier_v2.keras')
test_loss_CIFAR, test_acc_CIFAR = cnn_object_classifier_model_v2.evaluate(test_ds_CIFAR)
print(f"Test Accuracy: {test_acc_CIFAR * 100:.2f}%")

In [ ]:
cnn_object_classifier_inference_model_v2=keras.models.load_model(f"{CKPT_PATH}/cnn_object_classifier_v2.keras")

num_samples = 5
X_vis, y_true = [], []

for img, lbl in test_ds_CIFAR.unbatch().take(num_samples):
    X_vis.append(img)
    y_true.append([lbl])

X_vis = tf.stack(X_vis)
X_vis_norm = tf.cast(X_vis, tf.float32)
y_true = tf.stack(y_true)
print(X_vis.shape)
print(y_true.shape)

# Now plot
plot_predictions(
    cnn_object_classifier_inference_model_v2,
    X_vis_norm,
    X_vis.numpy(),
    y_true,
    img_shape=(32,32,3),
    class_names=CIFAR_CLASS,
    cmap=None,
    title="CIFAR CNN Predictions"
)


<a id="4"></a>
## **4. Kesimpulan dan Bacaan Lanjutan**
---

Model CNN berhasil diterapkan untuk klasifikasi gambar **CIFAR-10** dan **MNIST-Digits**.  
Dengan arsitektur yang digunakan, model mencapai akurasi yang cukup baik pada data uji, menandakan kemampuan jaringan dalam **menangkap pola spasial** dan **mengenali digit atau objek kecil**.  

Beberapa hal yang dapat dilakukan untuk meningkatkan performa model:  
- **Data augmentation**: rotasi, flipping, cropping, dan perubahan warna untuk memperluas variasi dataset.  
- **Arsitektur lebih kompleks**: menambahkan layer Conv2D, Dense, atau menggunakan residual connection.  
- **Regularisasi tambahan**: Dropout, L2 regularization, atau Batch Normalization untuk mengurangi overfitting.  
- **Optimasi training**: Early stopping, learning rate scheduling, dan penggunaan optimizer lainnya.  

Selamat belajar dan bereksperimen dengan Deep learning! 🚀

<a id="41"></a>
### Bacaan lanjutan
- [Early Stopping to Avoid Overtraining Neural Network Models](https://machinelearningmastery.com/early-stopping-to-avoid-overtraining-neural-network-models/)  
- [TensorFlow Datasets API Documentation](https://www.tensorflow.org/datasets/api_docs/python/tfds)  
- [TensorFlow CNN Guide](https://www.tensorflow.org/tutorials/images/cnn)  
- [CIFAR-10 Dataset Overview](https://www.cs.toronto.edu/~kriz/cifar.html)  
- [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf)  
- [Dropout Regularization in Deep Learning Models with Keras](https://machinelearningmastery.com/dropout-regularization-deep-learning-models-keras/)
